# NB3 · Sagittal dynamics, contact and sensor replay

Compare exact sticking dynamics to a unilateral compliant model and quantify a
slip detector using synthetic measurements. The regularized tangential law is
dissipative and permits nonzero creep when transmitting force. The prescribed
rolling mode is terminated on friction or unilateral failure.
[DUK v3 chapters 04](https://app.notion.com/p/4131f3cc033281f4bd54f24e8f92ce8e)
[and 05](https://app.notion.com/p/4131f3cc033281fdb641fadc2fe1ad72).

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from duk_core.modeling import load_leg_geometry_json

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'robot/config/physical/duk_leg_geometry.json').is_file())
geometry_path = repo / 'robot/config/physical/duk_leg_geometry.json'
fixture_path = repo / 'research/experiments/duk-v3/synthetic.json'
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture['schema_version'] == 1 and fixture['status'] == 'synthetic_research_only'
print('Working tree dirty:', bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo)))
print('Commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip())
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture['source'])

In [ ]:
from duk_core.modeling import build_planar_biped_model
from duk_core.modeling.planar_numeric import NumericPlanarBiped
from duk_core.modeling.rolling_reduction import lift_symmetric_rolling, symmetric_forward_wheel_input
from duk_core.modeling.contact_dynamics import CompliantContact, rigid_rolling
from duk_core.modeling.planar_experiment import simulate_planar
from duk_core.estimation.sagittal import (SlipGatedVelocity, absolute_wheel_rate, replay_velocity,
                                         complementary_pitch, ideal_planar_contact_estimate)

plant = NumericPlanarBiped(build_planar_biped_model(geometry), fixture['planar'])
p = fixture['planar_contact']
law = CompliantContact(p['stiffness'], p['damping'], p['slip_scale'])
q = np.deg2rad(95)
offset = plant.equivalent_upper_body(q).angle_from_vertical_rad
lift = lift_symmetric_rolling(geometry, 0, 0, 0, 0, q, mount_xz_m=(0., 0.), pitch_offset=offset)
initial = np.r_[lift.configuration, lift.velocity]
force = plant.masses.sum() * plant.parameters['g_m_s2'] / 2
static_effort = plant.static_hip_effort(lift.configuration, [[0., force], [0., force]])
d = plant.evaluate(lift.configuration, lift.velocity)
np.testing.assert_allclose(d.mass, d.mass.T, atol=1e-12)
assert np.min(np.linalg.eigvalsh(d.mass)) > 0
dt = 1e-6
potential_gradient = [(plant.evaluate(lift.configuration + dt * e, lift.velocity).potential
                      - plant.evaluate(lift.configuration - dt * e, lift.velocity).potential) / (2 * dt)
                      for e in np.eye(7)]
np.testing.assert_allclose(d.gravity, potential_gradient, atol=1e-8)
test_effort = np.array([.8, -.3, .5, -.2])
velocity = np.array([.1, -.1, .2, .5, -.25, .6, -.1])
kinematics = plant.kinematics(lift.configuration)
carrier_indices = [plant.names.index('BCO_' + side) for side in ('L', 'R')]
carrier_rate = kinematics.angular_rows[carrier_indices] @ velocity
np.testing.assert_allclose(velocity @ d.input @ test_effort,
                           velocity[3:5] @ test_effort[:2] + (velocity[5:] - carrier_rate) @ test_effort[2:])

def input_profile(amplitude):
    def command(t, state):
        u = symmetric_forward_wheel_input(amplitude)
        u[:2] = static_effort + p['hip_hold_kp'] * (q - state[3:5]) - p['hip_hold_kd'] * state[10:12]
        return u
    return command

times = np.linspace(0, .20, 81)
tolerances = dict(max_step=.003, rtol=1e-7, atol=1e-9)
high = np.full(2, p['high_friction'])
low = np.full(2, p['low_friction'])
rigid = simulate_planar(plant, initial, times, input_profile(.10), high, mode='rolling', **tolerances)
settled = initial.copy()
settled[1] -= force / law.stiffness
soft = simulate_planar(plant, settled, times, input_profile(.10), high, mode='compliant',
                       contact_law=law, **tolerances)
slide = simulate_planar(plant, settled, times, input_profile(1.8), low, mode='compliant',
                        contact_law=law, **tolerances)
assert not rigid_rolling(d, input_profile(1.8)(0., initial), low).admissible
assert rigid.rolling_feasible.all() and len(rigid.time) == len(times)
assert np.max(abs(rigid.slip)) < 1e-7
assert np.max(abs(soft.axle_speed - rigid.axle_speed)) < .025
assert np.max(abs(slide.slip)) > .05
assert np.all(slide.contact_forces[:, :2] >= 0)
assert np.all(slide.contact_forces[:, 2:] * slide.slip <= 1e-12)

airborne = initial.copy()
airborne[1] += .10
airborne[9] = .08
flight = simulate_planar(plant, airborne, np.linspace(0, .05, 16), lambda t, x: np.zeros(4),
                         low, mode='flight', **tolerances)
assert np.ptp(flight.energy) < 1e-7
print('Free-flight energy drift (J):', np.ptp(flight.energy))
print('High-friction rigid/compliant max axle speed difference (m/s):',
      np.max(abs(soft.axle_speed - rigid.axle_speed)))

# Schedule the samples away from the events so the terminal rows must come from the guards.
def increasing_torque(t, state):
    return input_profile(.10 + 8.5 * t)(t, state)

limited = simulate_planar(plant, initial, [0., .06, .12, .20], increasing_torque, low,
                          mode='rolling', **tolerances)
assert limited.termination == 'rolling_limit' and 0. < limited.time[-1] < .20
normal, tangent = np.split(limited.contact_forces[-1], 2)
assert np.min(low * normal - abs(tangent)) >= -2e-8

drop = initial.copy()
drop[1] += .10
landing = simulate_planar(plant, drop, [0., .05, .10, .20], lambda t, x: np.zeros(4),
                          low, mode='flight', **tolerances)
assert landing.termination == 'touchdown'
np.testing.assert_allclose(landing.time[-1], np.sqrt(.20 / plant.parameters['g_m_s2']), atol=1e-7)
print('Rolling limit and touchdown times (s):', limited.time[-1], landing.time[-1])


In [ ]:
rng = np.random.default_rng(p['sensor_seed'])
relative_measurement = slide.relative_spin + rng.normal(0, p['encoder_noise_rad_s'], slide.relative_spin.shape)
pitch_gyro = slide.state[:, 9] + rng.normal(0, .002, len(times))
wheel_rate = absolute_wheel_rate(relative_measurement, pitch_gyro[:, None], slide.carrier_rate)
surface = -plant.radius * wheel_rate.mean(axis=1)
independent_acceleration = slide.axle_acceleration.mean(axis=1) + rng.normal(0, p['acceleration_noise_m_s2'], len(times))
contact = slide.contact_forces[:, :2].sum(axis=1) > 0
estimate = replay_velocity(times, independent_acceleration, surface, contact,
                           SlipGatedVelocity(0., p['slip_threshold_m_s'], 2, .08))
estimated_speed = np.array([e.speed for e in estimate])
truth_speed = slide.axle_speed.mean(axis=1)
truth_slip = abs(slide.slip.mean(axis=1)) > p['slip_threshold_m_s']
detected = np.array([e.slip_detected for e in estimate])
first_true = times[np.flatnonzero(truth_slip)[0]]
first_detection = times[np.flatnonzero(detected)[0]]

theta = [slide.state[0, 2]]
accel_pitch = slide.state[:, 2] + rng.normal(0, .01, len(times))
for i, step in enumerate(np.diff(times), 1):
    theta.append(complementary_pitch(theta[-1], pitch_gyro[i], accel_pitch[i], step, .08))
estimated_ft, estimated_n = ideal_planar_contact_estimate(
    slide.applied[:, 2:], plant.inertias[list(plant.wheels)],
    slide.acceleration[:, 5:] + rng.normal(0, .05, slide.acceleration[:, 5:].shape),
    plant.radius, plant.masses.sum(), plant.parameters['g_m_s2'],
    slide.com_vertical_acceleration + rng.normal(0, .025, len(times)))
normal_error = estimated_n - slide.contact_forces[:, :2].sum(axis=1)
tangent_error = estimated_ft - slide.contact_forces[:, 2:]
rmse = lambda values: np.sqrt(np.mean(np.square(values)))
assert rmse(estimated_speed - truth_speed) < rmse(surface - truth_speed)
assert rmse(normal_error) < .3 and rmse(tangent_error) < .02
print('Sensor replay RMSE for pitch (rad), total normal and per-wheel tangent force (N):',
      rmse(theta - slide.state[:, 2]), rmse(normal_error), rmse(tangent_error))
print('True positive rate, false positive rate, detection delay (s):',
      detected[truth_slip].mean(), detected[~truth_slip].mean(), first_detection - first_true)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(times, truth_speed, label='True axle speed')
axes[0].plot(times, surface, label='Wheel odometry')
axes[0].plot(times, estimated_speed, label='Slip-gated estimate')
axes[1].plot(times, slide.slip.mean(axis=1), label='True slip')
axes[1].plot(times, [e.slip for e in estimate], label='Estimated slip')
axes[1].fill_between(times, 0, 1, where=detected, transform=axes[1].get_xaxis_transform(), alpha=.15)
for ax in axes:
    ax.set_ylabel('Speed (m/s)')
    ax.legend()
axes[1].set_xlabel('Time (s)')
plt.show()

The replay supplies an independently compensated axle acceleration and idealized
carrier-rate compensation. Sensor extrinsics, synchronization and long-duration
inertial drift are separate identification and validation work. The normal
estimate reports total force. The slip law resolves a finite-slip comparison
without identifying the static/kinetic transition or impact restitution of DUK.